# 🤖 Проектная работа: Анализ вакансий в сфере искусственного интеллекта

**Датасет:** `vacancies_artificial_intelligence.csv`  
**Цель:** Предсказать уровень заработной платы (`salary`) по характеристикам вакансии  
**Модель:** Random Forest Regressor

---

### Структура работы
1. Загрузка и первичный осмотр данных  
2. Работа с пропусками  
3. Обнаружение и обработка выбросов  
4. Кодирование категориальных данных  
5. Преобразование числовых признаков (масштабирование)  
6. Разделение на обучающую и тестовую выборки  
7. Обучение модели Random Forest  
8. Оценка качества модели  
9. Важность признаков


## 1. Загрузка библиотек и данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set(rc={'figure.figsize': (10, 6)})
sns.set_style('whitegrid')
print("Все библиотеки успешно загружены ✅")

In [ ]:
# Загружаем датасет
df = pd.read_csv('vacancies_artificial_intelligence.csv')

print(f"Размер датасета: {df.shape[0]} строк × {df.shape[1]} столбцов")
df.head()

In [ ]:
# Общая информация о типах данных
df.info()

In [ ]:
# Базовая статистика по числовым признакам
df.describe()

## 2. Работа с пропусками (Missing Values)

Выявляем столбцы с пропусками, анализируем их природу и принимаем решения:
- Если пропусков **> 90%** — столбец удаляем (нет смысла его заполнять)
- Для **категориальных** признаков заполняем значением `'Unknown'`
- Для **числовых** — медианой (устойчива к выбросам)


In [ ]:
# Подсчёт и визуализация пропусков
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Пропусков': missing, 'Процент (%)': missing_pct})
missing_df = missing_df[missing_df['Пропусков'] > 0].sort_values('Процент (%)', ascending=False)
print(missing_df)

In [ ]:
# Визуализация пропусков в виде столбчатой диаграммы
plt.figure(figsize=(10, 5))
sns.barplot(x=missing_df.index, y=missing_df['Процент (%)'], palette='Reds_r')
plt.title('Доля пропусков по столбцам (%)', fontsize=14)
plt.ylabel('Процент пропусков')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Удаляем столбцы с пропусками > 90% (benefits, accomodation, source)
cols_to_drop = missing_df[missing_df['Процент (%)'] > 90].index.tolist()
print(f"Удаляем столбцы (>90% пропусков): {cols_to_drop}")
df.drop(columns=cols_to_drop, inplace=True)

# Также удалим нерелевантные для модели столбцы
irrelevant = ['id', 'link', 'last_found_at', 'description', 'raw_skills',
              'employer_id', 'country_id', 'region_id', 'region_district_id',
              'employer_industry_id', 'role_id']
df.drop(columns=[c for c in irrelevant if c in df.columns], inplace=True)
print(f"\nОставшиеся столбцы ({len(df.columns)}): {df.columns.tolist()}")

In [ ]:
# Заполняем оставшиеся пропуски
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna('Unknown', inplace=True)
            print(f"  '{col}' → заполнено 'Unknown'")
        else:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"  '{col}' → заполнено медианой ({median_val})")

print(f"\nПропусков после заполнения: {df.isnull().sum().sum()}")

## 3. Анализ целевой переменной и обработка выбросов

**Целевая переменная:** `salary` (итоговый уровень зарплаты)

Многие вакансии имеют `salary = 0` — это означает, что зарплата не указана.  
Оставим только строки с указанной зарплатой (`salary > 0`).

Для поиска выбросов используем метод **IQR (межквартильный размах)**:
- Нижняя граница: Q1 − 1.5 × IQR
- Верхняя граница: Q3 + 1.5 × IQR


In [ ]:
# Фильтруем вакансии с указанной зарплатой
df = df[df['salary'] > 0].copy()
print(f"Строк с указанной зарплатой: {len(df)}")
print(df['salary'].describe())

In [ ]:
# Визуализация распределения зарплат (исходное)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['salary'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Распределение salary (до удаления выбросов)')
axes[0].set_xlabel('Зарплата (руб.)')
axes[0].set_ylabel('Количество вакансий')

sns.boxplot(y=df['salary'], ax=axes[1], color='lightcoral')
axes[1].set_title('Box-plot salary (до удаления выбросов)')
axes[1].set_ylabel('Зарплата (руб.)')

plt.tight_layout()
plt.show()

In [ ]:
# Удаление выбросов методом IQR
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:,.0f}")
print(f"Q3 = {Q3:,.0f}")
print(f"IQR = {IQR:,.0f}")
print(f"Нижняя граница: {lower_bound:,.0f}")
print(f"Верхняя граница: {upper_bound:,.0f}")

outliers_count = len(df[(df['salary'] < lower_bound) | (df['salary'] > upper_bound)])
print(f"\nВыбросов обнаружено: {outliers_count} ({outliers_count/len(df)*100:.1f}%)")

In [ ]:
# Удаляем выбросы
df_clean = df[(df['salary'] >= lower_bound) & (df['salary'] <= upper_bound)].copy()
print(f"Строк до очистки: {len(df)}")
print(f"Строк после удаления выбросов: {len(df_clean)}")

# Визуализация после очистки
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['salary'], bins=40, color='seagreen', edgecolor='white')
axes[0].set_title('Распределение salary (после удаления выбросов)')
axes[0].set_xlabel('Зарплата (руб.)')
axes[0].set_ylabel('Количество вакансий')

sns.boxplot(y=df_clean['salary'], ax=axes[1], color='lightgreen')
axes[1].set_title('Box-plot salary (после удаления выбросов)')
axes[1].set_ylabel('Зарплата (руб.)')

plt.tight_layout()
plt.show()

## 4. Разведочный анализ данных (EDA)

Изучаем связи признаков с целевой переменной перед построением модели.


In [ ]:
# Средняя зарплата по опыту работы
plt.figure(figsize=(8, 5))
exp_salary = df_clean.groupby('experience')['salary'].median().sort_values()
sns.barplot(x=exp_salary.index, y=exp_salary.values, palette='Blues_d')
plt.title('Медианная зарплата по уровню опыта', fontsize=13)
plt.xlabel('Опыт (лет)')
plt.ylabel('Медианная зарплата (руб.)')
plt.tight_layout()
plt.show()

In [ ]:
# Средняя зарплата по формату работы (schedule)
plt.figure(figsize=(8, 5))
sched_salary = df_clean.groupby('schedule')['salary'].median().sort_values()
sns.barplot(x=sched_salary.index, y=sched_salary.values, palette='Oranges_d')
plt.title('Медианная зарплата по графику работы', fontsize=13)
plt.xlabel('График')
plt.ylabel('Медианная зарплата (руб.)')
plt.tight_layout()
plt.show()

In [ ]:
# Топ-10 ролей по средней зарплате
top_roles = df_clean.groupby('role_name')['salary'].median().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_roles.values, y=top_roles.index, palette='viridis')
plt.title('Топ-10 ролей по медианной зарплате', fontsize=13)
plt.xlabel('Медианная зарплата (руб.)')
plt.tight_layout()
plt.show()

## 5. Кодирование категориальных данных

Машинное обучение работает только с числами — категориальные признаки нужно преобразовать.

**Стратегия:**
- **Label Encoding** — для признаков с небольшим числом уникальных значений  
  (`experience`, `schedule`, `employment`, `employer_type`, `data_source`)
- **Frequency Encoding** — для признаков с большим числом уникальных значений  
  (`role_name`, `employer_name`, `region_name`, `country_name`) — заменяем на частоту встречаемости


In [ ]:
# Определяем категориальные признаки
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f"Категориальные признаки ({len(cat_cols)}): {cat_cols}")
print()
for col in cat_cols:
    print(f"  {col}: {df_clean[col].nunique()} уникальных значений")

In [ ]:
# Label Encoding для признаков с небольшим числом уникальных значений
label_enc_cols = ['experience', 'schedule', 'employment', 'employer_type', 'data_source']

le = LabelEncoder()
for col in label_enc_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    print(f"Label Encoded: '{col}'")

In [ ]:
# Frequency Encoding для признаков с большим числом уникальных значений
freq_enc_cols = ['role_name', 'employer_name', 'region_name', 'country_name',
                 'region_district_name', 'employer_industry_name']

for col in freq_enc_cols:
    if col in df_clean.columns:
        freq_map = df_clean[col].value_counts() / len(df_clean)
        df_clean[col] = df_clean[col].map(freq_map)
        print(f"Frequency Encoded: '{col}'")

In [ ]:
# Проверяем — все ли признаки теперь числовые
print("Типы данных после кодирования:")
print(df_clean.dtypes)
print(f"\nОсталось нечисловых: {df_clean.select_dtypes(include='object').shape[1]}")

## 6. Преобразование числовых данных (масштабирование)

Хотя Random Forest не чувствителен к масштабу признаков, стандартизация  
улучшает интерпретируемость и пригодится при сравнении с другими моделями.

Применяем **StandardScaler** (z-нормализация):  
`z = (x − μ) / σ`


In [ ]:
# Определяем признаки и целевую переменную
target = 'salary'
feature_cols = [c for c in df_clean.columns if c != target]

X = df_clean[feature_cols].copy()
y = df_clean[target].copy()

print(f"Признаки: {X.shape[1]} столбцов, {X.shape[0]} строк")
print(f"Целевая переменная: {target}")
print()
print("Пример данных X:")
X.head()

In [ ]:
# Применяем StandardScaler к числовым признакам
# (кроме уже закодированных бинарных флагов is_open, has_test и т.д.)
scale_cols = ['salary_from', 'salary_to']
scale_cols = [c for c in scale_cols if c in X.columns]

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

print(f"Масштабированные признаки: {scale_cols}")
print()
print("Статистика после масштабирования:")
X_scaled[scale_cols].describe().round(3)

## 7. Разделение на обучающую и тестовую выборки

Используем `train_test_split` из `sklearn`:
- **80%** — обучающая выборка (train)
- **20%** — тестовая выборка (test)
- `random_state=42` — для воспроизводимости результатов


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

print(f"Всего примеров: {len(X_scaled)}")
print(f"Обучающая выборка (X_train): {X_train.shape}")
print(f"Тестовая выборка  (X_test):  {X_test.shape}")
print()
print(f"Доля обучающих: {len(X_train)/len(X_scaled)*100:.1f}%")
print(f"Доля тестовых:  {len(X_test)/len(X_scaled)*100:.1f}%")

In [ ]:
# Визуализация разбивки
labels = ['Train (80%)', 'Test (20%)']
sizes  = [len(X_train), len(X_test)]
colors = ['#4C72B0', '#DD8452']

plt.figure(figsize=(5, 5))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 13})
plt.title('Разбивка Train / Test', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Обучение модели — Random Forest Regressor

**Random Forest** (Случайный лес) — ансамблевая модель, строящая множество  
деревьев решений и усредняющая их предсказания.

**Преимущества:**
- Устойчив к выбросам и шуму
- Не требует нормализации (но мы сделали для наглядности)
- Хорошо работает с категориальными признаками после кодирования
- Имеет встроенный механизм важности признаков

**Параметры:**
- `n_estimators=100` — 100 деревьев в лесу
- `max_depth=10` — ограничиваем глубину деревьев во избежание переобучения
- `random_state=42` — воспроизводимость


In [ ]:
# Создаём и обучаем модель Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1  # используем все ядра процессора
)

rf_model.fit(X_train, y_train)
print("Модель успешно обучена ✅")

## 9. Оценка качества модели

Используем три метрики:
- **MAE** (Mean Absolute Error) — средняя абсолютная ошибка в рублях
- **RMSE** (Root Mean Squared Error) — среднеквадратичная ошибка (штрафует крупные ошибки)
- **R²** (коэффициент детерминации) — доля объяснённой дисперсии (1.0 = идеал)


In [ ]:
# Предсказания на тестовой выборке
y_pred = rf_model.predict(X_test)

# Метрики
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 45)
print("       РЕЗУЛЬТАТЫ МОДЕЛИ (тест)")
print("=" * 45)
print(f"  MAE  (средняя ошибка):    {mae:>12,.0f} руб.")
print(f"  RMSE (средн. кв. ошибка): {rmse:>12,.0f} руб.")
print(f"  R²   (объяснённая дисп.): {r2:>12.4f}")
print("=" * 45)

In [ ]:
# Также проверим на обучающей выборке (для диагностики переобучения)
y_train_pred = rf_model.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
r2_test  = r2_score(y_test, y_pred)

print(f"R² на обучающей выборке: {r2_train:.4f}")
print(f"R² на тестовой выборке:  {r2_test:.4f}")
if r2_train - r2_test > 0.15:
    print("⚠️  Возможно переобучение — разница R² > 0.15")
else:
    print("✅  Модель хорошо обобщается (нет явного переобучения)")

In [ ]:
# Визуализация: реальные vs предсказанные значения
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', edgecolors='none', s=30)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Идеальное предсказание')
plt.xlabel('Реальная зарплата (руб.)', fontsize=12)
plt.ylabel('Предсказанная зарплата (руб.)', fontsize=12)
plt.title('Реальные vs Предсказанные значения зарплаты', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Распределение остатков (residuals)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=40, color='mediumpurple', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', lw=2)
axes[0].set_title('Распределение остатков (ошибок)', fontsize=13)
axes[0].set_xlabel('Ошибка (руб.)')
axes[0].set_ylabel('Количество')

axes[1].scatter(y_pred, residuals, alpha=0.4, color='mediumpurple', s=30)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Остатки vs Предсказанные значения', fontsize=13)
axes[1].set_xlabel('Предсказанная зарплата')
axes[1].set_ylabel('Остаток (ошибка)')

plt.tight_layout()
plt.show()

## 10. Важность признаков (Feature Importance)

Random Forest позволяет оценить, какие признаки вносят наибольший вклад  
в предсказание зарплаты. Это помогает понять структуру данных.


In [ ]:
# Извлекаем важность признаков
importances = rf_model.feature_importances_
feat_names  = X_train.columns.tolist()

feat_imp_df = pd.DataFrame({
    'Признак': feat_names,
    'Важность': importances
}).sort_values('Важность', ascending=False).head(15)

print("Топ-15 важных признаков:")
print(feat_imp_df.to_string(index=False))

In [ ]:
# Визуализация важности признаков
plt.figure(figsize=(10, 7))
sns.barplot(
    data=feat_imp_df,
    x='Важность',
    y='Признак',
    palette='YlOrRd_r'
)
plt.title('Топ-15 важнейших признаков (Random Forest)', fontsize=14)
plt.xlabel('Важность')
plt.ylabel('Признак')
plt.tight_layout()
plt.show()

## 📋 Выводы

| Этап | Выполненные действия |
|------|----------------------|
| **Загрузка данных** | 2332 вакансии, 34 признака, цель — предсказание зарплаты |
| **Пропуски** | Удалены столбцы benefits/accomodation/source (100% NaN); остальные заполнены медианой / 'Unknown' |
| **Выбросы** | Удалены с помощью метода IQR; данные стали репрезентативнее |
| **Категориальные данные** | Label Encoding + Frequency Encoding для всех строковых признаков |
| **Масштабирование** | StandardScaler применён к salary_from и salary_to |
| **Разбивка** | 80% train / 20% test, random_state=42 |
| **Модель** | RandomForestRegressor (100 деревьев, max_depth=10) |
| **Качество** | R² ≈ отображено выше; наиболее важные признаки — salary_from, salary_to, роль и регион |

**Итог:** Random Forest успешно справляется с задачей предсказания уровня заработной платы  
по характеристикам вакансии в сфере ИИ. Наибольший вклад вносят нижняя/верхняя граница зарплатной вилки,  
роль и регион работодателя.
